<a href="https://colab.research.google.com/github/gabrielzeit/jax_learning/blob/main/JAX_learning_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JAX learning notebook

## 1. JAX basics - functional paradigm

### Immutability



In [2]:
import numpy as np
import jax.numpy as jnp

# Standard NumPy (Mutable)
arr_np = np.array([1, 2, 3])
arr_np[0] = 99
print("NumPy array:", arr_np)

# JAX (Immutable)
arr_jnp = jnp.array([1, 2, 3])
# Try to mutate it the same way:
arr_jnp[0] = 99

NumPy array: [99  2  3]


TypeError: JAX arrays are immutable and do not support in-place item assignment. Instead of x[idx] = y, use x = x.at[idx].set(y) or another .at[] method: https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html

In [3]:
# solution
arr_jnp = arr_jnp.at[0].set(99)
print("JAX NumPy array:", arr_jnp)

JAX NumPy array: [99  2  3]


### Pure Functions

In [4]:
# Task 2: Pure Functions
offset = 10
log_list = []

def process_data(x):
    log_list.append(x)
    print(f"Processing data point: {x}")
    return x + offset

result = process_data(5)
print(result)

Processing data point: 5
15


In [5]:
# solution
def jax_process_data(x, offset, jax_log_list):
    jax_log_list_app = jax_log_list + [x]
    return x + offset, jax_log_list_app

offset = 10
jax_log_list = []
jax_result = jax_process_data(5, offset, jax_log_list)
print(jax_result)

(15, [5])


### Stateless PRNG

In [6]:
import numpy as np
import jax

# NumPy (Stateful)
np.random.seed(42)
print("NumPy 1:", np.random.uniform())
print("NumPy 2:", np.random.uniform())

# JAX (Stateless)
key = jax.random.PRNGKey(42)
print("JAX 1:", jax.random.uniform(key))
print("JAX 2:", jax.random.uniform(key))

NumPy 1: 0.3745401188473625
NumPy 2: 0.9507143064099162
JAX 1: 0.48870957
JAX 2: 0.48870957


In [7]:
# solution
key = jax.random.PRNGKey(42)

# We want two random numbers, so we split the main key into 3 parts:
# 1 new carry key, and 2 subkeys for immediate use.
key, subkey1, subkey2 = jax.random.split(key, num=3)
print("JAX 1:", jax.random.uniform(subkey1))
print("JAX 2:", jax.random.uniform(subkey2))

JAX 1: 0.72766423
JAX 2: 0.6672406


## 2. Core Transformations

### jax.jit

In [8]:
import jax
import jax.numpy as jnp

@jax.jit
def normalize_data(x, apply_norm):
    if apply_norm:
        return (x - jnp.mean(x)) / jnp.std(x)
    else:
        return x

arr = jnp.array([1.0, 2.0, 3.0, 4.0])

# Try running this:
result = normalize_data(arr, True)

TracerBoolConversionError: Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function normalize_data at /tmp/ipykernel_22916/404161485.py:4 for jit. This concrete value was not available in Python because it depends on the value of the argument apply_norm.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError

In [ ]:
# solution

from functools import partial

@partial(jax.jit, static_argnums=(1,))
def normalize_data(x, apply_norm):
    if apply_norm:
        return (x - jnp.mean(x)) / jnp.std(x)
    else:
        return x

arr = jnp.array([1.0, 2.0, 3.0, 4.0])

result = normalize_data(arr, True)
print(result)

# OR
@partial(jax.jit, static_argnames=["apply_norm"])
def normalize_data(x, apply_norm):
    if apply_norm:
        return (x - jnp.mean(x)) / jnp.std(x)
    else:
        return x

arr = jnp.array([1.0, 2.0, 3.0, 4.0])

result = normalize_data(arr, True)
print(result)

### jax.vmap

In [ ]:
# Task: jax.vmap
def dot_product(v1, v2):
    return jnp.dot(v1, v2)

batch_v1 = jnp.array([
    [1., 2., 3., 4.],
    [5., 6., 7., 8.],
    [9., 10., 11., 12.]
])

single_v2 = jnp.array([1., 1., 1., 1.])

In [9]:
# solution
vectorized_dot = jax.vmap(fun=dot_product, in_axes=(0,None))(batch_v1, single_v2)
print(vectorized_dot)

NameError: name 'dot_product' is not defined

### jax.grad

In [10]:
import jax

# Task: jax.grad and jax.value_and_grad
def loss_fn(x):
    # f(x) = x^3 + 2x^2
    return x**3 + 2.0 * x**2

x_val = 3.0

print(loss_fn(x_val))

45.0


In [11]:
# solution
jax_grad = jax.grad(loss_fn, argnums=0)(x_val)
print("JAX grad: ", jax_grad)

fun, grad = jax.value_and_grad(loss_fn, argnums=0)(x_val)
print("JAX val and grad: ", fun, " and ", grad)

JAX grad:  39.0
JAX val and grad:  45.0  and  39.0


## 3. Structured Control Flow

### jax.lax

In [12]:
import jax
import jax.numpy as jnp

# A standard Python implementation of the ReLU activation function
@jax.jit
def relu_python(x):
    if x > 0:
        return x
    else:
        return 0.0

# This will fail with a TracerBoolConversionError
result = relu_python(jnp.array(-2.0))

TracerBoolConversionError: Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function relu_python at /tmp/ipykernel_22916/2554287819.py:5 for jit. This concrete value was not available in Python because it depends on the value of the argument x.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError

In [13]:
# solution
import jax
import jax.numpy as jnp

@jax.jit
def relu_jax(x):
    return jnp.where(x > 0.0, x, 0.0)

result = relu_jax(jnp.array(-2.0))
print(result)

0.0


### jax.lax.scan

In [14]:
import jax
import jax.numpy as jnp

# A standard Python loop computing a cumulative sum
def cumsum_python(arr):
    carry = 0.0
    outputs = []
    for x in arr:
        carry = carry + x
        outputs.append(carry)
    return carry, jnp.array(outputs)

arr = jnp.array([1.0, 2.0, 3.0, 4.0])

print("Python loop: ", cumsum_python(arr))

# Task: Write the step function for jax.lax.scan
def scan_step(carry, x):
    # TODO: Calculate the new carry and the current output
    # Return format must be: (new_carry, output)
    new_carry = carry + x
    output = new_carry
    return (new_carry, output)

# jax.lax.scan takes (step_function, initial_carry, array_to_loop_over)
final_carry, outputs = jax.lax.scan(scan_step, init=0.0, xs=arr)
print("Final carry:", final_carry)
print("Outputs:", outputs)

Python loop:  (Array(10., dtype=float32), Array([ 1.,  3.,  6., 10.], dtype=float32))
Final carry: 10.0
Outputs: [ 1.  3.  6. 10.]


## 4. Pytrees (Data Structures)

In [15]:
import jax
import jax.numpy as jnp

# A Pytree representing the parameters of a small neural network
model_params = {
    "layer1": {
        "weights": jnp.array([[1.0, 2.0], [3.0, 4.0]]),
        "bias": jnp.array([0.1, 0.2])
    },
    "layer2": {
        "weights": jnp.array([[5.0, 6.0]]),
        "bias": jnp.array([0.3])
    }
}
print(model_params)

{'layer1': {'weights': Array([[1., 2.],
       [3., 4.]], dtype=float32), 'bias': Array([0.1, 0.2], dtype=float32)}, 'layer2': {'weights': Array([[5., 6.]], dtype=float32), 'bias': Array([0.3], dtype=float32)}}


In [16]:
new_model_param = jax.tree.map(f=lambda x: x*10, tree=model_params)
print(new_model_param)

{'layer1': {'bias': Array([1., 2.], dtype=float32), 'weights': Array([[10., 20.],
       [30., 40.]], dtype=float32)}, 'layer2': {'bias': Array([3.], dtype=float32), 'weights': Array([[50., 60.]], dtype=float32)}}


## 5. Advanced Automatic Differentiation

In [17]:
import jax
import jax.numpy as jnp

# Task: Gradient Control
def loss_fn(w1, w2, x):
    # w1 represents a pre-trained layer we want to freeze.
    # w2 represents a new layer we are actively training.

    features = w1 * x

    # TODO: Modify 'features' using jax.lax.stop_gradient
    # so that gradients do NOT flow backward into w1.
    features = jax.lax.stop_gradient(features)

    predictions = w2 * features
    return jnp.sum(predictions ** 2)

w1 = jnp.array([1.0, 2.0])
w2 = jnp.array([0.5, 0.5])
x = jnp.array([2.0, 3.0])

# Compute gradients with respect to both w1 (argnums=0) and w2 (argnums=1)
grad_w1, grad_w2 = jax.grad(loss_fn, argnums=(0, 1))(w1, w2, x)

print("Grad w1:", grad_w1)
print("Grad w2:", grad_w2)

Grad w1: [0. 0.]
Grad w2: [ 4. 36.]


### Jacobians

In [18]:
import jax
import jax.numpy as jnp

# Task: Jacobians for multi-output functions
def multi_output_fn(x):
    # Takes an array of size 2, returns an array of size 2.
    # f(x) = [x[0]**2, x[0] * x[1]]
    return jnp.array([x[0]**2, x[0] * x[1]])

x_val = jnp.array([2.0, 3.0])

# TODO: Compute the Jacobian matrix using both jax.jacfwd and jax.jacrev.
fwd = jax.jacfwd(multi_output_fn)(x_val)
rev = jax.jacrev(multi_output_fn)(x_val)

print(fwd)
print(rev)

[[4. 0.]
 [3. 2.]]
[[4. 0.]
 [3. 2.]]


### JVP vs VJP

In [19]:
import jax
import jax.numpy as jnp

def my_fun(x):
    # f([x1, x2]) = [x1**2, x1*x2, x2**2]
    # 2 inputs, 3 outputs
    return jnp.array([x[0]**2, x[0]*x[1], x[1]**2])

x_val = jnp.array([2.0, 3.0])

# --- JVP (Forward-Mode) ---
# We want to push a "tangent" vector forward.
tangent_vector = jnp.array([1.0, 1.0])

# TODO 1: Compute the JVP.
# jax.jvp expects three arguments: the function, a tuple of primal inputs, and a tuple of tangent vectors.
# It returns a tuple: (primal_output, tangent_output)
primal, tangent = jax.jvp(my_fun, (x_val,), (tangent_vector,))
print("JVP primal and tangent: ", primal, " and " , tangent)


# --- VJP (Reverse-Mode) ---
# We want to pull a "cotangent" vector backward (like a gradient from a loss function).
cotangent_vector = jnp.array([1.0, 1.0, 1.0])

# TODO 2: Compute the VJP.
# Step A: jax.vjp takes the function and the primal inputs. It returns the primal output AND a special vjp_function.
primal_out, vjp_fn = jax.vjp(my_fun, x_val)

# Step B: Call that special vjp_function with a tuple containing your cotangent_vector to get the final input gradients.
input_grads, = vjp_fn(cotangent_vector)

print("VJP primal and input_grads: ", primal_out, " and ", input_grads)

JVP primal and tangent:  [4. 6. 9.]  and  [4. 5. 6.]
VJP primal and input_grads:  [4. 6. 9.]  and  [7. 8.]


### 6. Multi-Device Parallelism

### mesh, partition and sharding

In [2]:
import os
# Force JAX to simulate 8 CPU devices. MUST be run before any other JAX calls!
os.environ['XLA_FLAGS'] = '--xla_force_host_platform_device_count=8'

import jax
import jax.numpy as jnp
import numpy as np
from jax.sharding import Mesh, PartitionSpec, NamedSharding

devices = np.array(jax.devices())
print(f"Available devices: {len(devices)}")

# We have a large 1D array we want to distribute.
big_array = jnp.arange(8000)

# 1. Create a logical Mesh. We treat our 8 devices as a 1D grid and name that axis 'data'.
mesh = Mesh(devices, axis_names=('data',))

# TODO 2: Create a PartitionSpec.
# It tells JAX which mesh axis to map the array's dimensions to.
spec = PartitionSpec("data")

# TODO 3: Create a NamedSharding object that combines the mesh and the spec.
sharding = NamedSharding(mesh, spec)

# 4. Apply the sharding to the array.
sharded_array = jax.device_put(big_array, sharding)

# 5. Visualize how the data is physically split across devices.
jax.debug.visualize_array_sharding(sharded_array)

Available devices: 8


  CPU 0    CPU 1    CPU 2    CPU 3    CPU 4    CPU 5    CPU 6    CPU 7  
                                                                        

### shard_map and Collective Primitives


In [6]:
from jax.experimental.shard_map import shard_map
from jax import lax
from functools import partial


# We use the same mesh and spec from your previous block.
# out_specs=PartitionSpec() means the output will be a single scalar replicated on all devices.
@jax.jit
@partial(shard_map, mesh=mesh, in_specs=spec, out_specs=PartitionSpec())
def compute_global_sum(local_chunk):
    # 'local_chunk' is just the 1000-element slice of the array on THIS specific device.
    local_sum = jnp.sum(local_chunk)

    # TODO: Use a collective primitive to sum the 'local_sum' from all 8 devices.
    # The collective primitive needs to know which mesh axis to communicate across.
    global_sum = lax.psum(local_sum, axis_name="data")

    return global_sum

total = compute_global_sum(sharded_array)
print("Total global sum:", total)

Total global sum: 31996000


## 7. JAX Deep Learning Ecosystem

### Equinox

In [8]:
%pip install equinox

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.0 MB/s eta 0:00:00


In [9]:
import equinox as eqx
import jax
import jax.numpy as jnp

class SimpleModel(eqx.Module):
    linear: eqx.nn.Linear
    apply_relu: bool  # Static metadata, not a differentiable array!

    def __init__(self, key):
        self.linear = eqx.nn.Linear(in_features=2, out_features=1, key=key)
        self.apply_relu = True

    def __call__(self, x):
        out = self.linear(x)
        # Using jnp.where since Equinox handles static Python control flow dynamically!
        return jnp.where(self.apply_relu, jax.nn.relu(out), out)

key = jax.random.PRNGKey(42)
model = SimpleModel(key)
x_data = jnp.array([1.5, -0.5])
y_target = jnp.array([1.0])

def loss_fn(current_model, x, y):
    pred = current_model(x)
    return jnp.sum((pred - y)**2)

# If we run standard jax.value_and_grad, it tries to differentiate the boolean and fails:
loss, grads = jax.value_and_grad(loss_fn)(model, x_data, y_target)

TypeError: grad requires real- or complex-valued inputs (input dtype that is a sub-dtype of np.inexact), but got bool. If you want to use Boolean- or integer-valued inputs, use vjp or set allow_int to True.

In [11]:
# solution
loss, grads = eqx.filter_value_and_grad(loss_fn)(model, x_data, y_target)
print(loss, grads)

0.23219714 SimpleModel(
  linear=Linear(
    weight=f32[1,2], bias=f32[1], in_features=2, out_features=1, use_bias=True
  ),
  apply_relu=None
)


### Optax

In [12]:
%pip install optax

In [13]:
import optax

# 1. Define the strategy (stateless)
optimizer = optax.sgd(learning_rate=0.01)

# 2. Initialize the explicit state.
# We filter the model so the optimizer only tracks the differentiable arrays.
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

# --- A Single Training Step ---
# We already have our 'grads' from eqx.filter_value_and_grad

# TODO 1: Compute the updates and the new optimizer state using optimizer.update
# Hint: It expects the gradients, the current opt_state, and the model (params) as arguments.
params, static_metadata = eqx.partition(pytree=model, filter_spec=eqx.is_array)
updates, opt_state = optimizer.update(grads, opt_state, params)

# TODO 2: Apply the updates to the model to create a new, updated model using eqx.apply_updates
# Hint: It expects the current model and the updates as arguments.
model = eqx.apply_updates(model, updates)